In [21]:
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage 
from langchain_core.tools import InjectedToolArg, tool
import os 
import requests
import json
from typing import Annotated


load_dotenv()

True

### Tool Creation

Create tool to get exchange rate

In [ ]:
exchange_rate_api_key = os.getenv("EXCHANGE_RATE_API")
@tool
def get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    """
    This function fetch the currency conversion factor between base_currency and target_currency
    It use ISO 4217 Three Letter Currency Codes - e.g. USD for US Dollars, EUR for Euro etc
    """
    url = f"https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}"
    response = requests.get(url=url)
    return response.json()

#We have rate limit of 1500 request/month


In [65]:
#This is how the output will be

result = get_exchange_rate.invoke({'base_currency':"INR", 'target_currency':"USD"})
print(type(result))
result

<class 'dict'>


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782518402,
 'time_last_update_utc': 'Sat, 27 Jun 2026 00:00:02 +0000',
 'time_next_update_unix': 1782604802,
 'time_next_update_utc': 'Sun, 28 Jun 2026 00:00:02 +0000',
 'base_code': 'INR',
 'target_code': 'USD',
 'conversion_rate': 0.01059}

In [67]:
result['conversion_rate']

0.01059

create tool to multiply exchange rate with base currency to get final value

Note that, when we call an llm with: "what is the exchange rate of inr to usd, and give the final value of 10 inr in usd", then the model will call two function - "get_exchange_rate" and "convert", but to convert it wont have "conversion rate" at that point right. 

Basically it wont be able to get that current currency conversion rate before calling the first tool and hence wont have value of "conversion rate" to put into the "convert" function. 

So Now, we will not let the llm put value of "conversion rate" into "convert" function


_InjectedToolArg is used when a tool argument should not be generated by the LLM. Instead, your application injects that value at runtime._


In [34]:
@tool
def convert(base_currency_value:float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """ 
    Multiply the base currency value with conversion rate to get final currency value after conversion
    """
    return base_currency_value * conversion_rate


print(convert.args)
convert.args_schema.model_json_schema()

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'}}


{'description': 'Multiply the base currency value with conversion rate to get final currency value after conversion',
 'properties': {'base_currency_value': {'title': 'Base Currency Value',
   'type': 'number'},
  'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}},
 'required': ['base_currency_value', 'conversion_rate'],
 'title': 'convert',
 'type': 'object'}

In [23]:
#This is how the output will be 

result = convert.invoke({'base_currency_value':100, 'conversion_rate': 0.01059})
print(type(result))
result

<class 'float'>


1.059

### Tool Binding

In [43]:
#Create toolkit

class currency_exchange_toolkit:
    def get_tools():
        return [get_exchange_rate, convert]
    
toolkit = currency_exchange_toolkit.get_tools()

In [36]:
#Define LLM model
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Flash",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

In [44]:
#Bind Toolkit
tooled_llm = model.bind_tools(toolkit)

### Tool Calling

In [128]:
## Keeping chat history
messages = []

## SystemMessage
sys_msg_content = """
You are a currency conversion assistant.

Always follow this workflow:

1. If an exchange rate is needed, call get_exchange_rate.
2. Call both get_exchange_rate and convert together.
3. Do not perform multiplication yourself.
"""
sys_msg = SystemMessage(content=
                        sys_msg_content)
messages.append(sys_msg)

## HumangMessage
humam_msg = HumanMessage(content=
                         "What is the current conversion rate of us dollar to indian rupees. Also convert 10 dollar into rupees")
messages.append(humam_msg)

## AImessage
ai_message = tooled_llm.invoke(messages)
messages.append(ai_message)


In [129]:
messages

[SystemMessage(content='\nYou are a currency conversion assistant.\n\nAlways follow this workflow:\n\n1. If an exchange rate is needed, call get_exchange_rate.\n2. Call both get_exchange_rate and convert together.\n3. Do not perform multiplication yourself.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the current conversion rate of us dollar to indian rupees. Also convert 10 dollar into rupees', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure! Let me fetch the exchange rate and convert $10 to Indian Rupees at the same time.', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_exchange_rate', 'description': None}, 'id': 'call_00_c1RmhnXFTXQfU7LNfCR78641', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_01_6C3aKzOgSdWpikB18xS47252', 'type': 'function'}]}, response_metadata=

In [131]:
## Tool execution
for tool_call in ai_message.tool_calls:
    if tool_call['name']=="get_exchange_rate":
        tool_msg1 = get_exchange_rate.invoke(tool_call)
        conversion_rate = json.loads(tool_msg1.content)['conversion_rate']  #json.loads deserializes str containing json obj
        messages.append(tool_msg1)


    if tool_call['name']=="convert":
        tool_call['args']['conversion_rate'] = conversion_rate #append "conversion_rate" to arguments of convert
        tool_msg2 = convert.invoke(tool_call)
        messages.append(tool_msg2)

In [132]:
messages

[SystemMessage(content='\nYou are a currency conversion assistant.\n\nAlways follow this workflow:\n\n1. If an exchange rate is needed, call get_exchange_rate.\n2. Call both get_exchange_rate and convert together.\n3. Do not perform multiplication yourself.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the current conversion rate of us dollar to indian rupees. Also convert 10 dollar into rupees', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure! Let me fetch the exchange rate and convert $10 to Indian Rupees at the same time.', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_exchange_rate', 'description': None}, 'id': 'call_00_c1RmhnXFTXQfU7LNfCR78641', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_01_6C3aKzOgSdWpikB18xS47252', 'type': 'function'}]}, response_metadata=

In [134]:
## Final LLM call
result = tooled_llm.invoke(messages)

In [136]:
from IPython.display import Markdown, display
display(Markdown(result.content))

Here are the results:

### 💱 Current Exchange Rate
**1 US Dollar (USD)** = **94.45 Indian Rupees (INR)**

*(Data as of June 27, 2026)*

### 💵 Conversion
**$10 USD** = **₹944.54 INR**

So, $10 converts to approximately **₹944.54** based on the latest exchange rate.